# 06.0 Main comparison and thresholds

## Notebook overview
This notebook loads the saved main comparison outputs from notebook 05, identifies the strongest overall method and the strongest SSL method, then rebuilds only the required runtimes to run threshold analysis. It saves compact threshold summary tables, dense threshold sweep tables, and deployment-style figures so the practical trade-off between recall and false alarms can be reviewed without rerunning the full experiment suite.

## Purpose
- load the shared split manifest and confirm the earlier leakage checks are still clean
- load the saved main comparison table from notebook 05 and rank the candidate methods
- rebuild only the methods needed for threshold analysis rather than recomputing the full comparison
- set thresholds from validation-good score quantiles to stay aligned with the original project design
- save threshold summary tables and dense threshold sweep tables for later report or README use
- export compact figures that show practical deployment trade-offs between precision, recall, F1, and false positive rate


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Optional dependency install
#------------------------------------------------------------------------------
try:
    import faiss
    print("faiss already available")
except Exception:
    !pip -q install -U faiss-cpu


In [ ]:
#------------------------------------------------------------------------------
# 1.1 Imports and notebook helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import random
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

import faiss
from IPython.display import display


# Print a clean banner so long notebook output is easier to scan.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create the parent folder before saving an artefact.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Read a JSON file from disk.
def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)


# Return file size in MB if the path exists.
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


# Keep deterministic behaviour consistent with the earlier notebooks.
def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Build a stable category-specific seed from text.
def stable_seed_from_text(seed, text):
    key = f"{seed}_{text}".encode("utf-8")
    return int(hashlib.md5(key).hexdigest()[:8], 16)


# Reset tracked peak CUDA memory before one fit stage.
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


# Read the tracked peak CUDA memory in MB.
def get_peak_gpu_memory_mb():
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan


print_banner("Environment")
print("Python        :", sys.version.split()[0])
print("Torch         :", torch.__version__)
print("Torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count     :", torch.cuda.device_count())
for gpu_idx in range(torch.cuda.device_count()):
    print(f"GPU {gpu_idx} name   :", torch.cuda.get_device_name(gpu_idx))


# 2.0 Run settings

## Purpose
- define the threshold analysis settings in one place before any runtime building starts
- keep the notebook portable across Kaggle and local use
- keep output paths clean so the saved tables and figures can be reused directly in GitHub


In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "06_main_comparison_and_thresholds"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

IMG_SIZE = 224
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 16

BACKBONE_NAME = "resnet18"
FEATURE_LAYER_NAME = "layer3"
CORESET_RATIO = 0.05
MAX_TRAIN_PATCHES = 120_000
PATCH_SCORE_TOPK = 200
PADIM_DIM = 100
PADIM_EPS = 1e-4

THRESHOLD_POLICIES = {
    "high_recall": 0.95,
    "balanced": 0.98,
    "low_false_alarm": 0.995,
}
SWEEP_QUANTILES = np.round(np.linspace(0.90, 0.999, 20), 3)

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_01_ROOT = WORK_ROOT / "01_dataset_audit_and_splits" / RUN_MODE
NOTEBOOK_03_ROOT = WORK_ROOT / "03_autoencoder_baseline" / RUN_MODE
NOTEBOOK_04_ROOT = WORK_ROOT / "04_simclr_pretraining" / RUN_MODE
NOTEBOOK_05_ROOT = WORK_ROOT / "05_ssl_downstream_models" / RUN_MODE

SPLIT_MANIFEST_PATH = NOTEBOOK_01_ROOT / "json" / "split_manifest.json"
LEAKAGE_REPORT_PATH = NOTEBOOK_01_ROOT / "json" / "leakage_report.json"

AE_CHECKPOINTS_DIR = NOTEBOOK_03_ROOT / "checkpoints"
SSL_CHECKPOINTS_DIR = NOTEBOOK_04_ROOT / "checkpoints"

TABLE_MAIN_COMPARISON_MEAN_DEP_PATH = NOTEBOOK_05_ROOT / "tables" / "table_main_comparison_mean.csv"
TABLE_MAIN_COMPARISON_FULL_DEP_PATH = NOTEBOOK_05_ROOT / "tables" / "table_main_comparison_full.csv"
TABLE_EFFICIENCY_SUMMARY_DEP_PATH = NOTEBOOK_05_ROOT / "tables" / "table_efficiency_summary.csv"

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
THRESHOLD_SETTINGS_PATH = JSON_DIR / "threshold_settings.json"
THRESHOLD_TARGETS_PATH = JSON_DIR / "threshold_targets.json"
AVAILABLE_SSL_CHECKPOINTS_PATH = JSON_DIR / "available_ssl_checkpoints.json"

TABLE_THRESHOLDS_CATEGORY_PATH = TABLES_DIR / "table_thresholds_category.csv"
TABLE_THRESHOLDS_MEAN_PATH = TABLES_DIR / "table_thresholds_mean.csv"
TABLE_THRESHOLDS_FULL_PATH = TABLES_DIR / "table_thresholds_full.csv"

TABLE_THRESHOLD_SWEEP_CATEGORY_PATH = TABLES_DIR / "table_threshold_sweep_category.csv"
TABLE_THRESHOLD_SWEEP_MEAN_PATH = TABLES_DIR / "table_threshold_sweep_mean.csv"
TABLE_THRESHOLD_SWEEP_FULL_PATH = TABLES_DIR / "table_threshold_sweep_full.csv"

FIG_THRESHOLD_SUMMARY_PATH = FIGURES_DIR / "fig_threshold_summary.png"
FIG_THRESHOLD_POLICY_TRADEOFF_PATH = FIGURES_DIR / "fig_threshold_policy_tradeoff.png"
FIG_THRESHOLD_SWEEP_PATH = FIGURES_DIR / "fig_threshold_sweep.png"

RUN_METADATA = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "gpu_count": GPU_COUNT,
    "timestamp": datetime.now().isoformat(),
}
save_json_overwrite(RUN_METADATA, RUN_METADATA_PATH)

THRESHOLD_SETTINGS = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "categories_active": CATEGORIES_ACTIVE,
    "image_size": IMG_SIZE,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_test": BATCH_SIZE_TEST,
    "backbone_name": BACKBONE_NAME,
    "feature_layer_name": FEATURE_LAYER_NAME,
    "patchcore": {
        "coreset_ratio": CORESET_RATIO,
        "max_train_patches": MAX_TRAIN_PATCHES,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
    "padim": {
        "dim": PADIM_DIM,
        "eps": PADIM_EPS,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
    "threshold_policies": THRESHOLD_POLICIES,
    "sweep_quantiles": SWEEP_QUANTILES.tolist(),
}
save_json_overwrite(THRESHOLD_SETTINGS, THRESHOLD_SETTINGS_PATH)

print_banner("Run configuration")
print("RUN_MODE         :", RUN_MODE)
print("SEED             :", SEED)
print("DEVICE           :", DEVICE)
print("GPU_COUNT        :", GPU_COUNT)
print("FEATURE_LAYER    :", FEATURE_LAYER_NAME)
print("CORESET_RATIO    :", CORESET_RATIO)
print("PADIM_DIM        :", PADIM_DIM)
print("THRESHOLD_POLICIES:", THRESHOLD_POLICIES)
print("NOTEBOOK_ROOT    :", NOTEBOOK_ROOT)


# 3.0 Dataset and prior artefacts

## Purpose
- resolve the MVTec AD dataset path in a portable way
- load the split manifest and leakage report from notebook 01
- load the saved main comparison outputs from notebook 05
- check that the required SimCLR checkpoints are available for any SSL threshold target


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve dataset path and load prior notebook artefacts
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

required_paths = [
    SPLIT_MANIFEST_PATH,
    LEAKAGE_REPORT_PATH,
    TABLE_MAIN_COMPARISON_MEAN_DEP_PATH,
]
missing_required = [str(path) for path in required_paths if not Path(path).exists()]
if missing_required:
    raise FileNotFoundError(
        "One or more dependency artefacts are missing. Run notebooks 01 and 05 first.\n"
        + "\n".join(missing_required)
    )

split_manifest = read_json(SPLIT_MANIFEST_PATH)
leakage_report = read_json(LEAKAGE_REPORT_PATH)
table_main_comparison_mean = pd.read_csv(TABLE_MAIN_COMPARISON_MEAN_DEP_PATH)

table_main_comparison_full = None
if TABLE_MAIN_COMPARISON_FULL_DEP_PATH.exists():
    table_main_comparison_full = pd.read_csv(TABLE_MAIN_COMPARISON_FULL_DEP_PATH)

table_efficiency_summary = None
if TABLE_EFFICIENCY_SUMMARY_DEP_PATH.exists():
    table_efficiency_summary = pd.read_csv(TABLE_EFFICIENCY_SUMMARY_DEP_PATH)

leakage_failures = []
for key, value in leakage_report.items():
    if isinstance(value, (int, float)) and value != 0:
        leakage_failures.append((key, value))
    elif isinstance(value, list) and len(value) > 0:
        leakage_failures.append((key, f"{len(value)} records"))

if leakage_failures:
    raise RuntimeError(
        "Notebook 01 reported non-zero leakage checks. Fix notebook 01 before continuing.\n"
        + "\n".join([f"{k}: {v}" for k, v in leakage_failures])
    )

ssl_ckpt_map = {
    "mild": SSL_CHECKPOINTS_DIR / "simclr_mild_encoder.pt",
    "strong": SSL_CHECKPOINTS_DIR / "simclr_strong_encoder.pt",
}
available_ssl_ckpt_map = {tag: str(path) for tag, path in ssl_ckpt_map.items() if path.exists()}
save_json_overwrite(available_ssl_ckpt_map, AVAILABLE_SSL_CHECKPOINTS_PATH)

categories_from_manifest = list(split_manifest.keys())
CATEGORIES_ACTIVE = [cat for cat in CATEGORIES_ACTIVE if cat in categories_from_manifest]
if len(CATEGORIES_ACTIVE) == 0:
    raise RuntimeError("No active categories from the run configuration were found in the split manifest.")

available_categories = sorted([p.name for p in MVTEC_DIR.iterdir() if p.is_dir()])

print_banner("Loaded prior artefacts")
print("Dataset root                 :", MVTEC_DIR)
print("Split manifest path          :", SPLIT_MANIFEST_PATH)
print("Leakage report path          :", LEAKAGE_REPORT_PATH)
print("Main comparison mean path    :", TABLE_MAIN_COMPARISON_MEAN_DEP_PATH)
print("Main comparison full path    :", TABLE_MAIN_COMPARISON_FULL_DEP_PATH)
print("Efficiency summary path      :", TABLE_EFFICIENCY_SUMMARY_DEP_PATH)
print("Available SSL checkpoints    :", available_ssl_ckpt_map)
print("Active categories            :", CATEGORIES_ACTIVE)
print("Available dataset categories :", available_categories)

display(table_main_comparison_mean.sort_values("img_pr_auc", ascending=False))
if table_efficiency_summary is not None:
    display(table_efficiency_summary)


# 4.0 Shared datasets, transforms, metrics, and plotting helpers

## Purpose
- define the shared dataset wrappers used to rebuild only the selected threshold-analysis targets
- keep image preprocessing aligned with the earlier notebooks
- define the scoring and threshold metrics needed for the deployment-style threshold analysis


In [ ]:
#------------------------------------------------------------------------------
# 4.1 Shared transforms, dataset class, loaders, metrics, and plotting helpers
#------------------------------------------------------------------------------
tfm_imagenet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tfm_ae = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


# Load one RGB image.
def load_rgb(path):
    return Image.open(path).convert("RGB")


# Load one binary mask or return an all-zero mask for good images.
def load_mask(path):
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return torch.zeros((1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)

    mask = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    mask = (np.array(mask, dtype=np.float32) > 0).astype(np.float32)
    return torch.from_numpy(mask)[None, :, :]


# Return items in a consistent format across the shared split manifest.
class MvtecDataset(Dataset):
    def __init__(self, items, mode, img_tfm):
        self.items = items
        self.mode = mode
        self.img_tfm = img_tfm

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        if self.mode in ["train_good", "val_good"]:
            img_path = item
            label = 0
            mask_path = None
        else:
            img_path = item["img_path"]
            label = int(item["label"])
            mask_path = item.get("mask_path", None)

        image = self.img_tfm(load_rgb(img_path))
        mask = load_mask(mask_path)
        return image, int(label), mask, str(img_path)


# Build a DataLoader matched to the active runtime.
def make_loader(items, mode, input_kind, batch_size, shuffle):
    img_tfm = tfm_imagenet if input_kind == "imagenet" else tfm_ae
    dataset = MvtecDataset(items=items, mode=mode, img_tfm=img_tfm)

    loader_kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE == "cuda",
    }

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(**loader_kwargs)


# Build the train, validation, and test loaders for one category.
def make_split_loaders(category, input_kind):
    train_loader = make_loader(split_manifest[category]["train_good"], mode="train_good", input_kind=input_kind, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loader = make_loader(split_manifest[category]["val_good"], mode="val_good", input_kind=input_kind, batch_size=BATCH_SIZE_TEST, shuffle=False)
    test_loader = make_loader(split_manifest[category]["test"], mode="test", input_kind=input_kind, batch_size=BATCH_SIZE_TEST, shuffle=False)
    return train_loader, val_loader, test_loader


# Compute image-level ROC-AUC safely.
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Compute image-level PR-AUC safely.
def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)


# Flatten masks and heatmaps to compute pixel ROC-AUC.
def pixel_roc_auc(masks_list, heatmaps_list):
    if len(masks_list) == 0 or len(heatmaps_list) == 0:
        return np.nan

    y_true = np.concatenate([mask.reshape(-1) for mask in masks_list]).astype(int)
    y_score = np.concatenate([heat.reshape(-1) for heat in heatmaps_list]).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Evaluate one scoring function across the test set.
def eval_method(test_loader, score_fn):
    y_true = []
    y_score = []
    masks = []
    heats = []

    eval_start = time.time()
    eval_image_n = 0

    for images, labels, batch_masks, _ in test_loader:
        scores, heatmaps = score_fn(images)

        y_true.extend(labels.numpy().tolist())
        y_score.extend(scores.tolist())

        mask_np = batch_masks.squeeze(1).numpy()
        for idx in range(mask_np.shape[0]):
            masks.append(mask_np[idx])
            heats.append(heatmaps[idx])

        eval_image_n += images.shape[0]

    total_eval_sec = time.time() - eval_start

    return {
        "img_roc_auc": safe_roc_auc(y_true, y_score),
        "img_pr_auc": safe_pr_auc(y_true, y_score),
        "px_roc_auc": pixel_roc_auc(masks, heats),
        "sec_per_img": total_eval_sec / max(eval_image_n, 1),
        "n_eval_images": int(eval_image_n),
        "y_true": y_true,
        "y_score": y_score,
        "masks": masks,
        "heats": heats,
    }


# Collect one image-level score per item for threshold analysis.
def collect_image_scores(loader, score_fn):
    rows = []
    for images, labels, _, paths in loader:
        scores, _ = score_fn(images)
        for idx in range(len(paths)):
            rows.append({
                "path": paths[idx],
                "label": int(labels[idx].item()),
                "score": float(scores[idx]),
            })
    return pd.DataFrame(rows)


# Compute thresholded metrics at one cut-off.
def threshold_metrics(y_true, y_score, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    normal_mask = (y_true == 0)
    fp = np.sum((y_pred == 1) & normal_mask)
    n_normal = np.sum(normal_mask)
    fpr = fp / max(n_normal, 1)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
    }


# 5.0 Runtime builders for threshold analysis

## Purpose
- rebuild only the methods selected from the main comparison table
- keep the downstream settings aligned with the earlier notebooks
- support ImageNet, SSL, and autoencoder methods so the threshold analysis stays robust if rankings change in debug mode


In [ ]:
#------------------------------------------------------------------------------
# 5.1 Backbones, PatchCore, PaDiM, autoencoder, and runtime builders
#------------------------------------------------------------------------------
def get_resnet18_ssl():
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Identity()
    model.eval()
    return model


# Build the ImageNet backbone used in the baseline notebooks.
def get_resnet_imagenet():
    try:
        model = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        )
    except Exception as exc:
        raise RuntimeError(
            "ImageNet weights could not be loaded for resnet18. "
            "Run the notebook in an environment where torchvision pretrained weights are available."
        ) from exc

    model.fc = nn.Identity()
    model.eval()
    return model


# Register hooks so intermediate feature maps can be collected.
def make_feature_hooks(model, layer_names):
    features = {}
    handles = []

    def hook_factory(layer_name):
        def _hook(_, __, output):
            features[layer_name] = output
        return _hook

    for layer_name in layer_names:
        handle = getattr(model, layer_name).register_forward_hook(hook_factory(layer_name))
        handles.append(handle)

    return features, handles


# Remove hook handles after the runtime is no longer needed.
def remove_handles(handles):
    for handle in handles:
        handle.remove()


@torch.inference_mode()
def forward_get_feats(model, features, inputs, layer_names):
    _ = model(inputs)
    return [features[layer_name] for layer_name in layer_names]


# Flatten a feature map into patch rows.
def fmap_to_patches(feature_map):
    batch_n, channels, height, width = feature_map.shape
    return feature_map.permute(0, 2, 3, 1).reshape(batch_n, height * width, channels)


# Upsample and concatenate patch features from one or more layers.
def concat_patch_features(feature_maps):
    target_h, target_w = feature_maps[-1].shape[-2:]
    patch_list = []

    for feature_map in feature_maps:
        if feature_map.shape[-2:] != (target_h, target_w):
            feature_map = F.interpolate(
                feature_map,
                size=(target_h, target_w),
                mode="bilinear",
                align_corners=False,
            )
        patch_list.append(fmap_to_patches(feature_map))

    return torch.cat(patch_list, dim=-1)


# Build a FAISS L2 index for nearest-neighbour search.
def faiss_index_l2(array_2d):
    array_2d = np.asarray(array_2d, dtype=np.float32)
    index = faiss.IndexFlatL2(array_2d.shape[1])
    index.add(array_2d)
    return index


# Rebuild one saved SimCLR encoder checkpoint.
def load_ssl_encoder(ckpt_path):
    model = get_resnet18_ssl()
    state = torch.load(ckpt_path, map_location="cpu")

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    if isinstance(state, dict):
        clean_state = {}
        for key, value in state.items():
            clean_key = key.replace("module.", "", 1) if key.startswith("module.") else key
            clean_state[clean_key] = value
        state = clean_state

    model.load_state_dict(state, strict=True)
    model.eval()
    return model


# Collect normal train patch embeddings and keep a deterministic coreset subset.
def build_patch_bank(model, features, train_loader, layer_names, category, source_tag):
    patch_chunks = []
    total_patches = 0

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_maps = forward_get_feats(model, features, images, layer_names)
        patches = concat_patch_features(feature_maps).detach().cpu().numpy()
        patches = patches.reshape(-1, patches.shape[-1]).astype(np.float32)
        patch_chunks.append(patches)
        total_patches += patches.shape[0]

        if total_patches >= MAX_TRAIN_PATCHES:
            break

    bank_full = np.concatenate(patch_chunks, axis=0).astype(np.float32)

    if len(bank_full) > MAX_TRAIN_PATCHES:
        rng_cap = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{source_tag}_patch_bank_cap"))
        idx_cap = rng_cap.choice(len(bank_full), size=MAX_TRAIN_PATCHES, replace=False)
        bank_full = bank_full[idx_cap]

    keep_n = max(1, int(round(len(bank_full) * float(CORESET_RATIO))))
    keep_n = min(keep_n, len(bank_full))

    rng_core = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{source_tag}_patchcore_coreset"))
    coreset_idx = rng_core.choice(len(bank_full), size=keep_n, replace=False)
    bank = bank_full[coreset_idx]

    return bank


@torch.inference_mode()
def patchcore_scores(model, features, images, layer_names, index):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_maps = forward_get_feats(model, features, images, layer_names)
    patches = concat_patch_features(feature_maps).detach().cpu().numpy()

    batch_n, patch_n, feat_dim = patches.shape
    queries = patches.reshape(-1, feat_dim).astype(np.float32)
    dist2, _ = index.search(queries, 1)
    dist2 = dist2.reshape(batch_n, patch_n)

    feat_h, feat_w = feature_maps[-1].shape[-2:]
    heat = dist2.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Fit PaDiM Gaussian statistics from normal train patches.
def padim_fit(model, features, train_loader, layer_name, category, source_tag):
    all_feature_maps = []

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()
        all_feature_maps.append(feature_map)

    feature_map = torch.cat(all_feature_maps, dim=0)
    sample_n, channels, feat_h, feat_w = feature_map.shape

    dim = min(PADIM_DIM, channels)
    rng_dim = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{source_tag}_padim_dim"))
    dim_idx = rng_dim.choice(channels, size=dim, replace=False)

    feature_map = feature_map[:, dim_idx, :, :]
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(sample_n, feat_h * feat_w, dim).numpy()

    mu = feats_np.mean(axis=0)
    cov_inv = []

    eye = np.eye(dim, dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        patch_features = feats_np[:, patch_idx, :]
        cov = np.cov(patch_features, rowvar=False).astype(np.float32) + PADIM_EPS * eye
        cov_inv.append(np.linalg.inv(cov).astype(np.float32))

    cov_inv = np.stack(cov_inv, axis=0)

    return {
        "dim_idx": dim_idx.astype(np.int64),
        "mu": mu.astype(np.float32),
        "cov_inv": cov_inv.astype(np.float32),
        "H": int(feat_h),
        "W": int(feat_w),
        "D": int(dim),
    }


@torch.inference_mode()
def padim_scores(model, features, images, layer_name, stats):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()

    dim_idx = torch.tensor(stats["dim_idx"])
    feature_map = feature_map[:, dim_idx, :, :]

    batch_n, feat_dim, feat_h, feat_w = feature_map.shape
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(batch_n, feat_h * feat_w, feat_dim).numpy()

    mu = stats["mu"]
    cov_inv = stats["cov_inv"]

    heat = np.zeros((batch_n, feat_h * feat_w), dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        diff = feats_np[:, patch_idx, :] - mu[patch_idx]
        tmp = diff @ cov_inv[patch_idx]
        heat[:, patch_idx] = np.sum(tmp * diff, axis=1)

    heat = heat.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.dec(self.enc(x))


@torch.inference_mode()
def ae_scores(model, images):
    model.eval()
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))

    recon = model(images)
    err = (recon - images).pow(2).mean(dim=1)

    heatmaps = err.detach().cpu().numpy()
    flat = heatmaps.reshape(heatmaps.shape[0], -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heatmaps


# Translate the method label back into the settings needed to rebuild it.
def parse_method_name(method_name):
    if method_name == "imagenet_patchcore":
        return {"representation_source": "imagenet", "anomaly_head": "patchcore", "aug_strength": "none"}
    if method_name == "imagenet_padim":
        return {"representation_source": "imagenet", "anomaly_head": "padim", "aug_strength": "none"}
    if method_name == "autoencoder":
        return {"representation_source": "none", "anomaly_head": "autoencoder", "aug_strength": "none"}

    _, aug_strength, anomaly_head = method_name.split("_")
    return {"representation_source": "simclr", "anomaly_head": anomaly_head, "aug_strength": aug_strength}


# Build the loaders, backbone, and score function for one feature-based method.
def build_feature_method_runtime(category, representation_source, anomaly_head, aug_strength=None):
    train_loader, val_loader, test_loader = make_split_loaders(category, input_kind="imagenet")

    if representation_source == "imagenet":
        model = get_resnet_imagenet()
        ckpt_path = None
        source_tag = "imagenet"
    else:
        ckpt_path = Path(available_ssl_ckpt_map[aug_strength])
        model = load_ssl_encoder(ckpt_path)
        source_tag = aug_strength

    model = model.to(DEVICE)
    features, handles = make_feature_hooks(model, [FEATURE_LAYER_NAME])

    reset_peak_gpu_memory()
    fit_start = time.time()

    if anomaly_head == "patchcore":
        bank = build_patch_bank(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_names=[FEATURE_LAYER_NAME],
            category=category,
            source_tag=source_tag,
        )
        index = faiss_index_l2(bank)

        def score_fn(images):
            return patchcore_scores(model, features, images, [FEATURE_LAYER_NAME], index)

        fit_stats = {
            "feature_dim": int(bank.shape[1]),
            "memory_bank_n": int(bank.shape[0]),
            "memory_bank_mb": float(bank.nbytes / (1024 ** 2)),
        }

    elif anomaly_head == "padim":
        stats = padim_fit(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_name=FEATURE_LAYER_NAME,
            category=category,
            source_tag=source_tag,
        )

        def score_fn(images):
            return padim_scores(model, features, images, FEATURE_LAYER_NAME, stats)

        fit_stats = {
            "feature_dim": int(stats["D"]),
            "memory_bank_n": int(stats["H"] * stats["W"]),
            "memory_bank_mb": float((stats["mu"].nbytes + stats["cov_inv"].nbytes) / (1024 ** 2)),
        }

    else:
        remove_handles(handles)
        raise ValueError(f"Unsupported anomaly head: {anomaly_head}")

    fit_sec = float(time.time() - fit_start)
    peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "model": model,
        "features": features,
        "handles": handles,
        "score_fn": score_fn,
        "feature_dim": fit_stats["feature_dim"],
        "memory_bank_n": fit_stats["memory_bank_n"],
        "memory_bank_mb": fit_stats["memory_bank_mb"],
        "fit_sec": fit_sec,
        "peak_gpu_mem_mb": peak_gpu_mem_mb,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)) if ckpt_path is not None else np.nan,
    }


# Load the saved autoencoder checkpoint for one category.
def load_ae_checkpoint(category):
    ckpt_path = AE_CHECKPOINTS_DIR / f"ae_{category}.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"Autoencoder checkpoint was not found at {ckpt_path}. "
            "Run notebook 03 if you want to evaluate autoencoder thresholds."
        )

    model = SmallAE()
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state, strict=True)
    model.eval()
    return model, ckpt_path


# Rebuild the saved autoencoder and return a score function in the same format.
def build_autoencoder_runtime(category):
    _, val_loader, test_loader = make_split_loaders(category, input_kind="ae")
    model, ckpt_path = load_ae_checkpoint(category)
    model = model.to(DEVICE)

    def score_fn(images):
        return ae_scores(model, images)

    return {
        "train_loader": None,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "model": model,
        "features": None,
        "handles": [],
        "score_fn": score_fn,
        "feature_dim": np.nan,
        "memory_bank_n": np.nan,
        "memory_bank_mb": np.nan,
        "fit_sec": np.nan,
        "peak_gpu_mem_mb": np.nan,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)),
    }


# Route each method name to the matching runtime builder.
def build_method_runtime(category, method_name):
    meta = parse_method_name(method_name)

    if method_name == "autoencoder":
        runtime = build_autoencoder_runtime(category)
    else:
        runtime = build_feature_method_runtime(
            category=category,
            representation_source=meta["representation_source"],
            anomaly_head=meta["anomaly_head"],
            aug_strength=meta["aug_strength"] if meta["representation_source"] == "simclr" else None,
        )

    return {**meta, **runtime}


# 6.0 Load the main comparison and choose threshold targets

## Purpose
- use the saved mean comparison table from notebook 05 rather than rerunning the main comparison
- identify the strongest overall method and the strongest SSL method under the saved comparison
- save the selected threshold-analysis targets as a small JSON artefact for traceability


In [ ]:
#------------------------------------------------------------------------------
# 6.1 Select the target methods for threshold analysis
#------------------------------------------------------------------------------
rank_main = table_main_comparison_mean.dropna(subset=["img_pr_auc", "img_roc_auc"], how="all").copy()
if len(rank_main) == 0:
    raise RuntimeError(
        "The main comparison mean table does not contain valid ranking metrics. "
        "Rerun notebook 05 before continuing."
    )

rank_main = rank_main.sort_values(["img_pr_auc", "img_roc_auc", "px_roc_auc"], ascending=False).reset_index(drop=True)
BEST_OVERALL_METHOD = str(rank_main.iloc[0]["method"])

ssl_rank = rank_main[rank_main["representation_source"] == "simclr"].copy()
BEST_SSL_METHOD = None if len(ssl_rank) == 0 else str(ssl_rank.iloc[0]["method"])

threshold_targets = [{"target_name": "best_overall", "method": BEST_OVERALL_METHOD}]
if BEST_SSL_METHOD is not None and BEST_SSL_METHOD != BEST_OVERALL_METHOD:
    threshold_targets.append({"target_name": "best_ssl", "method": BEST_SSL_METHOD})

save_json_overwrite(threshold_targets, THRESHOLD_TARGETS_PATH)

print_banner("Threshold-analysis targets")
print("BEST_OVERALL_METHOD:", BEST_OVERALL_METHOD)
print("BEST_SSL_METHOD    :", BEST_SSL_METHOD)
print("Threshold targets  :", threshold_targets)

display(rank_main[[
    "method",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "img_pr_auc",
    "img_roc_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
]].head(10))


# 7.0 Threshold summary tables

## Purpose
- rebuild only the chosen methods and score validation-good and test images by category
- derive thresholds from validation-good quantiles to stay aligned with the project design
- save both category-level and mean threshold summary tables for later reuse


In [ ]:
#------------------------------------------------------------------------------
# 7.1 Run threshold analysis and save summary tables
#------------------------------------------------------------------------------
threshold_rows = []

print_banner("Running threshold summary analysis")
for target in threshold_targets:
    target_name = target["target_name"]
    method_name = target["method"]

    print("\n" + "-" * 90)
    print("TARGET:", target_name, "| METHOD:", method_name)
    print("-" * 90)

    for category in CATEGORIES_ACTIVE:
        runtime = build_method_runtime(category, method_name)

        df_val_scores = collect_image_scores(runtime["val_loader"], runtime["score_fn"])
        df_test_scores = collect_image_scores(runtime["test_loader"], runtime["score_fn"])

        val_scores = df_val_scores["score"].values.astype(float)
        y_true = df_test_scores["label"].values.astype(int)
        y_score = df_test_scores["score"].values.astype(float)

        for policy_name, quantile in THRESHOLD_POLICIES.items():
            threshold_value = float(np.quantile(val_scores, quantile))
            mets = threshold_metrics(y_true, y_score, threshold_value)

            threshold_rows.append({
                "target_name": target_name,
                "method": method_name,
                "representation_source": runtime["representation_source"],
                "anomaly_head": runtime["anomaly_head"],
                "aug_strength": runtime["aug_strength"],
                "category": category,
                "policy": policy_name,
                "quantile": quantile,
                "threshold": threshold_value,
                **mets,
            })

        remove_handles(runtime["handles"])
        del runtime["model"]
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        print(
            category,
            {
                "n_val": len(df_val_scores),
                "n_test": len(df_test_scores),
            }
        )

df_thresholds_category = pd.DataFrame(threshold_rows)

mean_threshold_cols = ["precision", "recall", "f1", "fpr"]
df_thresholds_mean = (
    df_thresholds_category
    .groupby(["target_name", "method", "representation_source", "anomaly_head", "aug_strength", "policy"], as_index=False)[mean_threshold_cols]
    .mean(numeric_only=True)
)
df_thresholds_mean["category"] = "MEAN"
df_thresholds_mean["quantile"] = df_thresholds_mean["policy"].map(THRESHOLD_POLICIES)
df_thresholds_mean["threshold"] = np.nan

ordered_threshold_cols = [
    "target_name",
    "method",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "category",
    "policy",
    "quantile",
    "threshold",
    "precision",
    "recall",
    "f1",
    "fpr",
]

df_thresholds_category = df_thresholds_category[ordered_threshold_cols]
df_thresholds_mean = df_thresholds_mean[ordered_threshold_cols]
df_thresholds_full = pd.concat([df_thresholds_category, df_thresholds_mean], ignore_index=True)

save_csv_overwrite(df_thresholds_category, TABLE_THRESHOLDS_CATEGORY_PATH)
save_csv_overwrite(df_thresholds_mean, TABLE_THRESHOLDS_MEAN_PATH)
save_csv_overwrite(df_thresholds_full, TABLE_THRESHOLDS_FULL_PATH)

print_banner("Threshold summary")
display(df_thresholds_mean.sort_values(["target_name", "policy"]))
print("Saved:", TABLE_THRESHOLDS_CATEGORY_PATH)
print("Saved:", TABLE_THRESHOLDS_MEAN_PATH)
print("Saved:", TABLE_THRESHOLDS_FULL_PATH)


# 8.0 Dense threshold sweep

## Purpose
- run a denser sweep of validation-derived quantiles so the trade-off curve is smoother
- save category-level and mean sweep tables for deployment-style plots
- make it easier to compare recall against false positive rate across the chosen target methods


In [ ]:
#------------------------------------------------------------------------------
# 8.1 Run the dense threshold sweep and save sweep tables
#------------------------------------------------------------------------------
sweep_rows = []

print_banner("Running dense threshold sweep")
for target in threshold_targets:
    target_name = target["target_name"]
    method_name = target["method"]

    print("\n" + "-" * 90)
    print("TARGET:", target_name, "| METHOD:", method_name)
    print("-" * 90)

    for category in CATEGORIES_ACTIVE:
        runtime = build_method_runtime(category, method_name)

        df_val_scores = collect_image_scores(runtime["val_loader"], runtime["score_fn"])
        df_test_scores = collect_image_scores(runtime["test_loader"], runtime["score_fn"])

        val_scores = df_val_scores["score"].values.astype(float)
        y_true = df_test_scores["label"].values.astype(int)
        y_score = df_test_scores["score"].values.astype(float)

        for quantile in SWEEP_QUANTILES:
            threshold_value = float(np.quantile(val_scores, quantile))
            mets = threshold_metrics(y_true, y_score, threshold_value)

            sweep_rows.append({
                "target_name": target_name,
                "method": method_name,
                "representation_source": runtime["representation_source"],
                "anomaly_head": runtime["anomaly_head"],
                "aug_strength": runtime["aug_strength"],
                "category": category,
                "quantile": float(quantile),
                "threshold": threshold_value,
                **mets,
            })

        remove_handles(runtime["handles"])
        del runtime["model"]
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

df_threshold_sweep_category = pd.DataFrame(sweep_rows)

mean_sweep_cols = ["precision", "recall", "f1", "fpr"]
df_threshold_sweep_mean = (
    df_threshold_sweep_category
    .groupby(["target_name", "method", "representation_source", "anomaly_head", "aug_strength", "quantile"], as_index=False)[mean_sweep_cols]
    .mean(numeric_only=True)
)
df_threshold_sweep_mean["category"] = "MEAN"
df_threshold_sweep_mean["threshold"] = np.nan

ordered_sweep_cols = [
    "target_name",
    "method",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "category",
    "quantile",
    "threshold",
    "precision",
    "recall",
    "f1",
    "fpr",
]

df_threshold_sweep_category = df_threshold_sweep_category[ordered_sweep_cols]
df_threshold_sweep_mean = df_threshold_sweep_mean[ordered_sweep_cols]
df_threshold_sweep_full = pd.concat([df_threshold_sweep_category, df_threshold_sweep_mean], ignore_index=True)

save_csv_overwrite(df_threshold_sweep_category, TABLE_THRESHOLD_SWEEP_CATEGORY_PATH)
save_csv_overwrite(df_threshold_sweep_mean, TABLE_THRESHOLD_SWEEP_MEAN_PATH)
save_csv_overwrite(df_threshold_sweep_full, TABLE_THRESHOLD_SWEEP_FULL_PATH)

print_banner("Threshold sweep summary")
display(df_threshold_sweep_mean.tail(20))
print("Saved:", TABLE_THRESHOLD_SWEEP_CATEGORY_PATH)
print("Saved:", TABLE_THRESHOLD_SWEEP_MEAN_PATH)
print("Saved:", TABLE_THRESHOLD_SWEEP_FULL_PATH)


# 9.0 Threshold figures

## Purpose
- export one compact summary figure for the named threshold policies
- export a deployment-style policy trade-off plot
- export a smoother recall versus false positive rate plot from the dense threshold sweep


In [ ]:
#------------------------------------------------------------------------------
# 9.1 Save threshold summary and trade-off figures
#------------------------------------------------------------------------------
policy_order = ["high_recall", "balanced", "low_false_alarm"]
target_order = [item["target_name"] for item in threshold_targets]

plot_summary_df = df_thresholds_mean.copy()
plot_summary_df["policy"] = pd.Categorical(plot_summary_df["policy"], categories=policy_order, ordered=True)
plot_summary_df["target_name"] = pd.Categorical(plot_summary_df["target_name"], categories=target_order, ordered=True)
plot_summary_df = plot_summary_df.sort_values(["target_name", "policy"])

fig = plt.figure(figsize=(14, 10))

summary_metrics = [
    ("precision", "Mean precision"),
    ("recall", "Mean recall"),
    ("f1", "Mean F1"),
    ("fpr", "Mean false positive rate"),
]

for subplot_idx, (metric_name, title_text) in enumerate(summary_metrics, start=1):
    ax = plt.subplot(2, 2, subplot_idx)

    width = 0.35 if max(len(target_order), 1) > 1 else 0.55
    x = np.arange(len(policy_order))

    for target_idx, target_name in enumerate(target_order):
        target_df = plot_summary_df[plot_summary_df["target_name"] == target_name].copy()
        target_df = target_df.set_index("policy").reindex(policy_order)
        offset = (target_idx - (len(target_order) - 1) / 2.0) * width
        values = target_df[metric_name].values.astype(float)
        bars = ax.bar(x + offset, values, width=width, label=str(target_name))

        for bar, value in zip(bars, values):
            if not np.isnan(value):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    f"{value:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

    ax.set_title(title_text)
    ax.set_xticks(x)
    ax.set_xticklabels(policy_order, rotation=15)
    if metric_name != "fpr":
        ax.set_ylim(0, 1.05)
    ax.legend()

plt.tight_layout()
ensure_parent(FIG_THRESHOLD_SUMMARY_PATH)
plt.savefig(FIG_THRESHOLD_SUMMARY_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_THRESHOLD_SUMMARY_PATH)

tradeoff_df = df_thresholds_mean.copy()
tradeoff_df["policy"] = pd.Categorical(tradeoff_df["policy"], categories=policy_order, ordered=True)
tradeoff_df["target_name"] = pd.Categorical(tradeoff_df["target_name"], categories=target_order, ordered=True)
tradeoff_df = tradeoff_df.sort_values(["target_name", "policy"])

fig = plt.figure(figsize=(10, 7))
for target_name in target_order:
    target_df = tradeoff_df[tradeoff_df["target_name"] == target_name].copy()
    plt.plot(target_df["fpr"].values, target_df["recall"].values, marker="o", label=str(target_name))

    for _, row in target_df.iterrows():
        plt.annotate(
            str(row["policy"]),
            (float(row["fpr"]), float(row["recall"])),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )

plt.xlabel("False positive rate")
plt.ylabel("Recall")
plt.title("Threshold policy trade-off")
plt.xlim(left=0)
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
ensure_parent(FIG_THRESHOLD_POLICY_TRADEOFF_PATH)
plt.savefig(FIG_THRESHOLD_POLICY_TRADEOFF_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_THRESHOLD_POLICY_TRADEOFF_PATH)

sweep_plot_df = df_threshold_sweep_mean.copy()
sweep_plot_df["target_name"] = pd.Categorical(sweep_plot_df["target_name"], categories=target_order, ordered=True)
sweep_plot_df = sweep_plot_df.sort_values(["target_name", "quantile"])

fig = plt.figure(figsize=(10, 7))
for target_name in target_order:
    target_df = sweep_plot_df[sweep_plot_df["target_name"] == target_name].copy()
    plt.plot(target_df["fpr"].values, target_df["recall"].values, marker="o", markersize=3, label=str(target_name))

plt.xlabel("False positive rate")
plt.ylabel("Recall")
plt.title("Dense threshold sweep")
plt.xlim(left=0)
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
ensure_parent(FIG_THRESHOLD_SWEEP_PATH)
plt.savefig(FIG_THRESHOLD_SWEEP_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_THRESHOLD_SWEEP_PATH)


# 10.0 Final review

## Purpose
- print the saved artefact paths in one place
- make it easy to confirm that the threshold analysis stage completed cleanly
- surface the saved mean tables that will be used most often in the report or README


In [ ]:
#------------------------------------------------------------------------------
# 10.1 Final saved artefacts
#------------------------------------------------------------------------------
print_banner("Saved artefacts")
print("Run metadata path              :", RUN_METADATA_PATH)
print("Threshold settings path        :", THRESHOLD_SETTINGS_PATH)
print("Threshold targets path         :", THRESHOLD_TARGETS_PATH)
print("Available checkpoints path     :", AVAILABLE_SSL_CHECKPOINTS_PATH)
print("Threshold category table path  :", TABLE_THRESHOLDS_CATEGORY_PATH)
print("Threshold mean table path      :", TABLE_THRESHOLDS_MEAN_PATH)
print("Threshold full table path      :", TABLE_THRESHOLDS_FULL_PATH)
print("Sweep category table path      :", TABLE_THRESHOLD_SWEEP_CATEGORY_PATH)
print("Sweep mean table path          :", TABLE_THRESHOLD_SWEEP_MEAN_PATH)
print("Sweep full table path          :", TABLE_THRESHOLD_SWEEP_FULL_PATH)
print("Threshold summary figure path  :", FIG_THRESHOLD_SUMMARY_PATH)
print("Policy trade-off figure path   :", FIG_THRESHOLD_POLICY_TRADEOFF_PATH)
print("Dense sweep figure path        :", FIG_THRESHOLD_SWEEP_PATH)

display(df_thresholds_mean.sort_values(["target_name", "policy"]))
display(df_threshold_sweep_mean.head(20))
